# Day 69: Deep Learning Mini Project
## Cats vs Dogs — End-to-End Image Classification

---

# THE CHALLENGE

Build the best cat-vs-dog classifier you can. Use everything from Days 61-68.

**Requirements:**
1. Load and preprocess image data
2. Build a CNN (or use transfer learning)
3. Train with proper callbacks
4. Evaluate with metrics and visualizations
5. Achieve >80% accuracy (bonus: >90%)

**You can use:** CNN from scratch OR transfer learning OR both and compare!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import tensorflow_datasets as tfds

# Download Cats vs Dogs (auto-downloads!)
(raw_train, raw_test), info = tfds.load(
    'cats_vs_dogs', split=['train[:80%]', 'train[80%:]'],
    as_supervised=True, with_info=True
)

print(f"Dataset: {info.features['label'].names}")
print(f"Total images: {info.splits['train'].num_examples:,}")

# Preprocessing
IMG_SIZE = 150
BATCH = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = raw_train.map(preprocess).batch(BATCH).prefetch(tf.data.AUTOTUNE)
test_ds = raw_test.map(preprocess).batch(BATCH).prefetch(tf.data.AUTOTUNE)

print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print("Data pipeline ready!")


In [ ]:
# Visualize samples
class_names = ['Cat', 'Dog']
plt.figure(figsize=(14, 7))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i+1)
        plt.imshow(images[i])
        plt.title(class_names[labels[i].numpy()], fontsize=12, fontweight='bold')
        plt.axis('off')
plt.suptitle('Cats vs Dogs Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Build CNN from scratch
cnn_model = keras.Sequential([
    # Block 1
    layers.Conv2D(32, 3, activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D(2),
    # Block 2
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    # Block 3
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(2),
    # Block 4
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(2),
    # Classifier
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.summary()


In [ ]:
# Train CNN
early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

print("Training CNN from scratch...")
history_cnn = cnn_model.fit(train_ds, epochs=15, validation_data=test_ds,
                             callbacks=[early_stop], verbose=1)


In [ ]:
# Plot CNN results
cnn_loss, cnn_acc = cnn_model.evaluate(test_ds, verbose=0)
print(f"CNN Test Accuracy: {cnn_acc:.4f} ({cnn_acc*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ['accuracy', 'loss']):
    ax.plot(history_cnn.history[metric], label=f'Train {metric}', linewidth=2)
    ax.plot(history_cnn.history[f'val_{metric}'], label=f'Val {metric}', linewidth=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel(metric.capitalize())
    ax.set_title(f'CNN {metric.capitalize()}'); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# Visualize predictions
plt.figure(figsize=(14, 10))
for images, labels in test_ds.take(1):
    probs = cnn_model.predict(images, verbose=0)
    preds = (probs > 0.5).astype(int).flatten()
    for i in range(8):
        plt.subplot(2, 4, i+1)
        plt.imshow(images[i])
        true = class_names[labels[i].numpy()]
        pred = class_names[preds[i]]
        conf = probs[i][0] if preds[i] == 1 else 1 - probs[i][0]
        color = 'green' if true == pred else 'red'
        plt.title(f'True: {true}\nPred: {pred} ({conf:.1%})', color=color, fontsize=10, fontweight='bold')
        plt.axis('off')
plt.suptitle('CNN Predictions — Green=Correct, Red=Wrong', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Bonus: Compare with Transfer Learning (MobileNetV2)
from tensorflow.keras import applications

base = applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

tl_model = keras.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])
tl_model.compile(optimizer=keras.optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
print("Training transfer learning model...")
history_tl = tl_model.fit(train_ds, epochs=5, validation_data=test_ds, callbacks=[early_stop], verbose=1)
tl_loss, tl_acc = tl_model.evaluate(test_ds, verbose=0)

print(f"\n=== FINAL COMPARISON ===")
print(f"CNN from scratch:       {cnn_acc:.3f} ({cnn_acc*100:.1f}%)")
print(f"Transfer Learning:      {tl_acc:.3f} ({tl_acc*100:.1f}%)")

if tl_acc > cnn_acc:
    print(f"\nTransfer Learning wins by +{(tl_acc-cnn_acc)*100:.1f}%!")
    print("This is why professionals use pre-trained models.")
else:
    print(f"\nCNN from scratch wins — impressive for a custom architecture!")


---

## Project Complete!

You built TWO image classifiers end-to-end:
1. A CNN from scratch (4 convolutional blocks)
2. A transfer learning model (MobileNetV2)

**What you learned:**
- Loading image datasets with tf.data pipeline
- Building CNNs for binary classification
- Using pre-trained models for better results
- Visualizing predictions and errors
- Comparing different approaches

**What to try on your own:**
- Add data augmentation (RandomFlip, RandomRotation)
- Try more pre-trained models (ResNet, EfficientNet)
- Fine-tune the transfer learning model
- Deploy the best model with Flask

**Tomorrow:** Final Day — DL Advanced Topics + Career Path!